# Hadoop環境のアップグレード — 確認と確定

更新後のデータとサービスの動作を確認し、HDFSの更新を確定する。

## *Operation Note*

*This is a cell for your own recording. ここに経緯を記述*

# 操作対象クラスタの設定

D10で作成したinventoryとgroup_varsを使用する。

In [ ]:
target_group = 'hadoop_all_testcluster'
target_group

In [ ]:
!ansible {target_group} -m ping

In [ ]:
import yaml
with open('group_vars/' + target_group) as configuration:
    kerberos_enabled = 'kerberos_realm' in yaml.safe_load(configuration)
kerberos_enabled

# 管理操作の認証

Kerberos構成の場合、HDFSの管理操作を行うホストでhdfsユーザーのkeytabを使って認証する。

In [ ]:
if kerberos_enabled:
    !ansible 'hadoop_namenode:hadoop_client' -l {target_group} --become --become-user hdfs -m shell -a 'kinit -kt /etc/hadoop/conf/hdfs.keytab hdfs/$(hostname -f)'

# Playbookの準備

In [ ]:
import os
import tempfile
work_dir = tempfile.mkdtemp()
work_dir

使用するNotebookと同じ版のPlaybookを取得する。

In [ ]:
!git clone https://github.com/NII-cloud-operation/Literate-computing-Hadoop.git {work_dir}/hadoop
!git -C {work_dir}/hadoop log -1

In [ ]:
playbook_dir = os.path.join(work_dir, 'hadoop/playbooks')
playbook_dir

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs haadmin -getAllServiceState' -l {target_group}

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs dfsadmin -report' -l {target_group}

Live datanodesが対象の全台であり、Missing blocks・Blocks with corrupt replicas・Under replicated blocksがすべて0であることを確認する。回復中なら時間をおいて再確認する。

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs fsck /' -l {target_group}

Status: HEALTHYであることを確認する。

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs dfsadmin -rollingUpgrade query' -l {target_group}

# 既存データの照合

In [ ]:
verification_path = '/dataset/iris/iris.data'
verification_path

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a "hdfs dfs -checksum {verification_path}" -l {target_group}

更新前のチェックサムと照合する。HBaseの読み書き、MapReduce・Sparkの実行結果も確認する。

# 更新の確定

finalize後はHDFSのrolling rollbackができなくなる。確認結果から、確定してよいと判断したら実行する。

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs dfsadmin -rollingUpgrade finalize' -l {target_group}

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs dfsadmin -rollingUpgrade query' -l {target_group}

no rolling upgrade in progressとなることを確認する。

# NameNodeを通常のサービス管理へ戻す

現在のActive/Standbyを指定する。Standbyから戻し、フェイルオーバー後に旧Activeを戻す。

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs haadmin -getAllServiceState' -l {target_group}

In [ ]:
active_namenode_host = 'XXX.XXX.XXX.71'
standby_namenode_host = 'XXX.XXX.XXX.70'
active_namenode_id = 'nn2'
standby_namenode_id = 'nn1'
(active_namenode_host, standby_namenode_host, active_namenode_id, standby_namenode_id)

Activeでのfinalize完了後も、Standbyが編集ログを取り込むまで待つ。両NameNodeに確定が反映されてから停止する。

In [ ]:
namenode_rpc_port = 8020
namenode_rpc_port


In [ ]:
namenode_http_port = 50070
namenode_http_port


In [ ]:
namenode_https_port = 50470
namenode_ca_file = '/etc/ssl/certs/ca-certificates.crt'
(namenode_https_port, namenode_ca_file)


In [ ]:
if kerberos_enabled:
    !ansible hadoop_namenode --become --become-user hdfs -m shell -a 'curl --fail --no-progress-meter --negotiate -u : --cacert {namenode_ca_file} "https://$(hostname):{namenode_https_port}/jmx?qry=Hadoop:service=NameNode,name=NameNodeInfo"' -l {target_group}
else:
    !ansible hadoop_namenode --become --become-user hdfs -m shell -a 'curl --fail --no-progress-meter "http://$(hostname):{namenode_http_port}/jmx?qry=Hadoop:service=NameNode,name=NameNodeInfo"' -l {target_group}

両NameNodeのRollingUpgradeStatusがnullになったことを確認する。未反映なら時間をおいて再確認する。

In [ ]:
!ansible {standby_namenode_host} --become -m service -a "name=hadoop-hdfs-zkfc state=stopped" -l {target_group}

In [ ]:
!ansible {standby_namenode_host} --become -a "systemctl stop hadoop-hdfs-namenode-upgrade" -l {target_group}

In [ ]:
!ansible {standby_namenode_host} --become --become-user hdfs -a "hdfs namenode -metadataVersion" -l {target_group}

HDFS Image VersionとSoftware format versionが一致していることを確認する。

In [ ]:
!ansible {standby_namenode_host} --become -m service -a "name=hadoop-hdfs-namenode state=started" -l {target_group}

In [ ]:
!ansible {standby_namenode_host} -m wait_for -a "host={standby_namenode_host} port={namenode_rpc_port} timeout=180" -l {target_group}

In [ ]:
!ansible {standby_namenode_host} --become -m service -a "name=hadoop-hdfs-zkfc state=started" -l {target_group}

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a "hdfs haadmin -getServiceState {standby_namenode_id}" -l {target_group}

standbyに戻ったことを確認する。

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a "hdfs haadmin -failover {active_namenode_id} {standby_namenode_id}" -l {target_group}

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs haadmin -getAllServiceState' -l {target_group}

In [ ]:
!ansible {active_namenode_host} --become -m service -a "name=hadoop-hdfs-zkfc state=stopped" -l {target_group}

In [ ]:
!ansible {active_namenode_host} --become -a "systemctl stop hadoop-hdfs-namenode-upgrade" -l {target_group}

In [ ]:
!ansible {active_namenode_host} --become --become-user hdfs -a "hdfs namenode -metadataVersion" -l {target_group}

HDFS Image VersionとSoftware format versionが一致していることを確認する。

In [ ]:
!ansible {active_namenode_host} --become -m service -a "name=hadoop-hdfs-namenode state=started" -l {target_group}

In [ ]:
!ansible {active_namenode_host} -m wait_for -a "host={active_namenode_host} port={namenode_rpc_port} timeout=180" -l {target_group}

In [ ]:
!ansible {active_namenode_host} --become -m service -a "name=hadoop-hdfs-zkfc state=started" -l {target_group}

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a "hdfs haadmin -getServiceState {active_namenode_id}" -l {target_group}

standbyに戻ったことを確認する。

In [ ]:
!ansible hadoop_client --become --become-user hdfs -a 'hdfs dfsadmin -report' -l {target_group}

Live datanodesが対象の全台であり、Missing blocks・Blocks with corrupt replicas・Under replicated blocksがすべて0であることを確認する。回復中なら時間をおいて再確認する。

In [ ]:
!ansible hadoop_namenode --become -a "systemctl status --no-pager hadoop-hdfs-namenode" -l {target_group}

両NameNodeが通常のサービスで稼働し、利用者の処理が継続していることを確認する。[基盤パッケージの更新](O12k_Hadoop%20-%20Update%20platform%20packages.ipynb)へ進む。